# nano Beatrix — the Canon bodies through the v4 curriculum, beside the twins

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_canon_train.ipynb)

Two bodies with **Canon layers** go through the v4 train's curriculum on the seeds still open, beside the twins on the same seeds. A Canon layer mixes every byte's vector with the three before it, channel by channel, and adds the result back (a causal convolution of width four with a residual; Allen-Zhu, arXiv 2512.17351). Each block has one before its mixer and one before its bank. The weights are born at zero, so each body starts as its parent.

| arm | the body |
|---|---|
| `polluxc-both` | Pollux (splat hubs at blocks 0, 2 and 4 with the rotary address, rotary softmax at 1, 3 and 5) with Canon layers |
| `altdc-both` | Pollux's layout with dual hubs (each hub keeps two address masses, one rotary and one plain) and Canon layers |
| `castor-both`, `pollux-both` | the twins: the reference trains, no Canon layers |

Why these two: on the eight-record frames of stage A, both Canon bodies learned the keyed lookup strongly on seeds 1, 2 and 4 (copy 0.91-0.93) and neither did on seed 3. Pollux's layout with dual hubs but no Canon layers learned it strongly on one seed of four. On seeds 1 and 2, both Canon bodies' whole trains ended at Castor's level of recall.

The queue runs in phases. Each phase is a curriculum over its own chains, run to its end before the next phase starts:

| phase | chains | what it reads |
|---|---|---|
| 1 | Castor and Pollux, seed 4 | the reference for the two Canon bodies' seed-4 trains (trained on another card; their blocks are on the Hub) |
| 2 | both Canon bodies, seed 3 | neither learned the lookup at stage A on this seed: can the long window rescue it? The chains continue from their stage-A blocks on the Hub |
| 3 | Castor and Pollux, seed 3 | the reference for phase 2 |

The stages are the v4 train's, unchanged:

| stage | window x rows a step | the rows | a block | what decides the next block |
|---|---|---|---|---|
| A, the copy | 1,024 x 32 | three rows in four framed (frames of 32 / 128 / 256 bytes, eight records a lookup frame, a frame nested in 0.6 of the thinking frames, a key defined twice in 0.3 of the rows), one in four a plain document | 12,000 steps | the copy formed: move on; not formed: one more block, then move on, marked |
| B, the long window and the count | 4,096 x 8 | the same with frames to 1,024 bytes and sixteen records a frame | 4,000 steps | another block while the reads of records more than 512 bytes back still fall by more than 0.1 bits over the block's last 800 steps; at most five blocks |
| the anneal | 4,096 x 8 | the plain diet: nesting 0.3, no key defined twice, four records a frame, frames to 1,024 | 4,000 steps | one block |

**Length:** C2 measures each arm on the card in hand and prints the queue's length. On an RTX PRO 6000, expect about two to three hours. The Canon layers cost about half again as much a step as their parents in this first, unfused form.

**Shipping and resuming:** every finished block ships to the training repo as it lands, and the decisions and boundary table ship at the end of each phase under `bed/curriculum/`. A `STOP` file in `/content/out` starts no new block. Rerunning C3 continues, and a new session continues from the blocks already on the Hub.

**Token:** put a Hugging Face token with write access to the training repo in this notebook's secrets as `HF_TOKEN` before C1. Without one, everything runs and nothing ships.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.25"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 25), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner ships through the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished block ships here (its record and its weights); tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (every pass used these 160 MB, the same stream order)
NAME = 'nano_canon_train'                              # the train's name: its decisions and boundary tables ship as bed/curriculum/*_<NAME>_p<phase>.*
PHASES = [                                             # (arms, seeds): each phase a curriculum over its own chains, run to its end in order
    ('castor-both,pollux-both', '4'),     # the twins on seed 4: the reference for the Canon bodies' seed-4 trains
    ('altdc-both,polluxc-both', '3'),     # the Canon bodies on seed 3: neither learned the lookup at stage A; can the long window rescue it
                                          # (dual hubs + Canon reached the continuation only; Pollux + Canon nothing in two blocks)
    ('castor-both,pollux-both', '3'),     # the twins on seed 3: the reference for the phase above
]
FORM = 'sliced'                    # the frame form of the v4 train (the fourth pass's pick): every close free; Pollux best sliced
FORMS = dict(plain={}, sized=dict(sized=1), sliced=dict(sized=1, sliced=32))
BASE = dict(
    register='caption',            # the register of every pass
    frames='dyn', plant=0.75,      # three rows in four framed, one in four a plain document (an all-framed diet made a frame specialist)
    ask='any', tags='mode',        # the training rows ask any question; a craft trained on one strict question learns a stricter grammar
    gauge_asks='nested,overlap',   # the gauge asks the nested and the redefined-key questions strictly at every block's end (192 rows each)
    log_every=100,                 # the progress line; the curve's counts make the far reads' 800-step windows exact
    **FORMS[FORM])
STAGES = [                         # the v4 train's stages, unchanged (colab/nano_v4_train.ipynb)
    dict(name='A', grid=dict(ctx=1024, mb=32, steps=12000, sizes='32,128,256', frame_records=8, nest=0.6, overlap=0.3),
         rule='formed', max_blocks=2),              # the copy formed: move on; not formed: one more block, then move on, marked
    dict(name='B', grid=dict(ctx=4096, mb=8, steps=4000, sizes='32,128,256,512,1024', frame_records=16, nest=0.6, overlap=0.3),
         rule='far_fall', fall=0.1, window=800, band='512', max_blocks=5),   # another block while the far reads fall > 0.1 bits
    dict(name='anneal', grid=dict(ctx=4096, mb=8, steps=4000, sizes='32,128,256,512,1024', frame_records=4, nest=0.3, overlap=0.0),
         rule=None, max_blocks=1),                  # the diet of record at the end (a diet change, never a learning-rate change)
]
SMOKE_STEPS = 200                  # C2's short runs of each arm on this card for the price line; never shipped
print('decision block:', NAME, '| form', FORM, '| phases', ' ; '.join(f'{a} x seeds {s}' for a, s in PHASES),
      '| stages', ' -> '.join(s['name'] for s in STAGES), '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, every block each phase may run, a short run of every arm at each stage's rows on this card (the price
# line), the push probe.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_curriculum as bc, bed_runner

# the data: the corpus and the held-out head from fineweb-edu (tokenless; a few minutes; the lines carry the time left)
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'))

# the plan: every block each phase's chains may run, and each chain's next block (blocks already on disk or on the Hub are read, not rerun)
NEXT = {}
for i, (arms, seeds) in enumerate(PHASES, 1):
    print(f'--- phase {i}: {arms} x seeds {seeds}')
    NEXT.update(bc.run(STAGES, BASE, arms, seeds, out=OUT, push=REPO, name=f'{NAME}_p{i}', dry=True)['next'])

# the price line: SMOKE_STEPS of every arm at each stage shape's rows (window x rows a step), from birth, without the strict-question
# gauges (the step rate is the progress lines'); the records stay local under their own short-run names and are never shipped
ARMS = list(dict.fromkeys(a for arms, _ in PHASES for a in arms.split(',')))
SHAPES = list(dict.fromkeys((int(st['grid']['ctx']), int(st['grid']['mb'])) for st in STAGES))
rate = {}
for arm in ARMS:
    for ctx, mb in SHAPES:
        st = next(s for s in STAGES if (int(s['grid']['ctx']), int(s['grid']['mb'])) == (ctx, mb))
        spec = dict(dict(BASE, **st['grid']), arms=arm, seeds='1', steps=SMOKE_STEPS, log_every=20, gauge_asks='')
        res = bed_runner.run(spec, out=OUT)
        assert res['failed'] is None, f'the short run of {arm} failed: read the lines above before the train'
        rate[(arm, ctx, mb)] = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]['tok_s'] * 1e3

def _left(craft, nxt):
    # the training seconds a chain has left from its next block: (fewest, most), as the reads decide
    arm = craft.split(' s')[0]
    lo = hi = 0.0
    if nxt is None:
        return lo, hi
    name = nxt.rstrip('0123456789')
    k = int(nxt[len(name):] or 1)
    started = False
    for st in STAGES:
        started = started or st['name'] == name
        if not started:
            continue
        ctx, mb = int(st['grid']['ctx']), int(st['grid']['mb'])
        t = int(st['grid']['steps']) * ctx * mb / rate[(arm, ctx, mb)]
        most = (st['max_blocks'] if st['rule'] else 1) - (k - 1 if st['name'] == name else 0)
        lo, hi = lo + t, hi + t * max(most, 1)
    return lo, hi

print('this card (tokens a second): ' + '; '.join(f'{a} ' + ', '.join(f'{rate[(a, c, m)] / 1e3:.0f}k at {c:,} x {m}' for c, m in SHAPES) for a in ARMS))
tot_lo = tot_hi = 0.0
for craft, nxt in NEXT.items():
    lo, hi = _left(craft, nxt)
    tot_lo, tot_hi = tot_lo + lo, tot_hi + hi
    print(f'  {craft}: next {nxt or "nothing (done)"}; {lo / 3600:.2f} to {hi / 3600:.2f} h of training left')
print(f'-> the queue trains {tot_lo / 3600:.1f} to {tot_hi / 3600:.1f} h one block at a time, as the reads decide, plus about a minute a'
      f' block for the start, the gauges and the ship')

# the push probe (the ship-complete law): a write to the training repo and a read-back BEFORE any training. Tokenless = disarmed.
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = f'bed/preflight/push_probe_{NAME}.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': {f'{a}_{c}x{m}': v for (a, c, m), v in rate.items()}, 'form': FORM, 'phases': PHASES,
                       'stages': STAGES, 'base': BASE}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message=f'{NAME}: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')

In [ ]:
# C3 (T) — THE TRAIN: the phases in order; inside a phase the chains go in rounds (each chain one block, then the next round), one block
# at a time on the card; each block's record decides its chain's next block by its stage's rule (a BOUNDARY line says what it read and
# what follows); every block ships to REPO in the background as it lands (a Hub fault retries with a doubling wait and never stops the
# train); a STOP file in OUT starts no new block. Rerun the cell to continue: the decisions replay from the records (on disk, or on the
# Hub in a new session) and only the missing blocks run. An interrupt loses only the running block (the bed has no mid-run checkpoint).
from geolip.nano_beatrix import bed_curriculum as bc
for i, (arms, seeds) in enumerate(PHASES, 1):
    print(f'=== phase {i}/{len(PHASES)}: {arms} x seeds {seeds}')
    res = bc.run(STAGES, BASE, arms, seeds, out=OUT, push=REPO if (HF_TOKEN and PUSH_OK) else '', name=f'{NAME}_p{i}')
    print({k: v for k, v in res.items() if k != 'table'})
    if res['failed'] is not None or res['stopped']:
        print('the train paused here: read the lines above; rerun this cell to continue')
        break

In [ ]:
# C4 — THE BOUNDARY TABLES: one row per block of every chain (what its record read, and what followed), from the records on disk or
# on the Hub, phase by phase. Run any time; it trains nothing.
from IPython.display import Markdown, display
from geolip.nano_beatrix import bed_curriculum as bc
for i, (arms, seeds) in enumerate(PHASES, 1):
    display(Markdown(f'### Phase {i}: {arms} x seeds {seeds}\n\n' + bc.report(STAGES, BASE, arms, seeds, out=OUT, repo=REPO)))

### Changing the train

Edit the decision block in C1, rerun C1 (the install is cached), then C2 and C3. Blocks already on disk or on the Hub are read and never rerun. A new seed starts clean.

| to | the edit in C1 |
|---|---|
| more seeds of both Canon bodies and the twins | add `('altdc-both,polluxc-both,castor-both,pollux-both', '5,6')` to `PHASES` |
| one body only | `('polluxc-both', '3')` in place of the pair |
| a longer stage B, or a stricter settling | `max_blocks=8`; `fall=0.05` in stage B |

On a smaller card, use `mb=4` at 4,096 and `mb=16` at 1,024 (the step counts stay; a block takes longer). The session folder is the Colab disk: the Hub holds the records, the weights, and at the end of each phase the decisions and the boundary table.